# 23 — Knowledge Management

Ask it a question. It searches a 5-document corpus, selects the most relevant precedents, and drafts a `KnowledgeBrief` that cites them by name — no hallucinated references.

In [ ]:
!pip install -q langchain-openai langchain-core python-dotenv

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from pydantic import BaseModel, Field

class Precedent(BaseModel):
    title: str = Field(description='Document title.')
    excerpt: str = Field(description='Relevant extract.')
    relevance_reason: str = Field(description='Why relevant.')
    source_id: str = Field(description='Document ID.')

class KnowledgeBrief(BaseModel):
    query: str = Field(description='Original query.')
    precedents: list[Precedent] = Field(description='Retrieved precedents.')
    synthesis: str = Field(description='Synthesis citing precedents by title.')
    gaps: list[str] = Field(description='Gaps not covered by precedents.')

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

CORPUS = [
    {'id': 'DOC-001', 'title': 'EMEA SaaS Go-to-Market Playbook (2022)', 'summary': 'Partner-led motion outperformed direct sales in mid-market by 2.4x across UK, Germany, France.'},
    {'id': 'DOC-002', 'title': 'FinTech Regulatory Compliance (2023)', 'summary': 'PSD2 and GDPR gap analysis for payment processing client. 14-week engagement.'},
    {'id': 'DOC-003', 'title': 'Manufacturing Digital Transformation -- Phase 1 (2023)', 'summary': 'IoT rollout across 3 facilities. Change management was the critical path, not technology.'},
    {'id': 'DOC-004', 'title': 'Investor Readiness Review -- Series B (2024)', 'summary': 'SaaS HR-tech gaps: weak unit economics, no profitability path in 18 months.'},
    {'id': 'DOC-005', 'title': 'Strategic Sourcing Review (2022)', 'summary': '$14M savings identified across 12 procurement categories for a logistics provider.'},
]
_INDEX = '\n'.join(f'[{d["id"]}] {d["title"]}: {d["summary"]}' for d in CORPUS)

_RETRIEVE = SystemMessage(f'Select the 2-4 most relevant documents for the query. Return a KnowledgeBrief with empty synthesis and gaps.\n\nCorpus:\n{_INDEX}')
_SYNTHESISE = SystemMessage('Synthesise a grounded response. Cite precedents by title. Note gaps.')

def run(query):
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(KnowledgeBrief)
    retrieved = llm.invoke([_RETRIEVE, {'role': 'user', 'content': f'Query: {query}'}])
    precedent_text = '\n\n'.join(f'[{p.source_id}] {p.title}\n{p.excerpt}' for p in retrieved.precedents)
    brief = llm.invoke([_SYNTHESISE, {'role': 'user', 'content': f'Query: {query}\n\nPrecedents:\n{precedent_text}'}])
    brief.precedents = retrieved.precedents
    return brief

In [ ]:
brief = run('We are pitching a SaaS company on EMEA expansion. What does prior work say about go-to-market strategy?')
print(f'Precedents ({len(brief.precedents)}):')
for p in brief.precedents:
    print(f'  [{p.source_id}] {p.title}')
    print(f'    {p.relevance_reason}')
print(f'\nSynthesis:\n{brief.synthesis}')
if brief.gaps:
    print(f'\nGaps:')
    for g in brief.gaps:
        print(f'  - {g}')

## Exercise

Add a sixth document to the corpus about a regulatory engagement. Then ask a query that should hit both the regulatory doc and the EMEA playbook. Verify the synthesis cites both by title.